# jev-lanes in five minutes

Run all cells in order. This tutorial uses real LangGraph execution and scripted providers: no credentials, network calls or model costs. Thresholds are illustrative.

From the checkout, install with `uv sync --locked --dev --group notebook`, then select `.venv/bin/python` in your notebook editor. To verify without an editor, run `uv run --locked --group notebook python examples/execute_notebook.py`.

In [ ]:
from typing import Any, TypedDict

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command
from pydantic import ValidationError

from jev_lanes import Classification, JevRouter, Lanes, RouteTrace
from jev_lanes.testing import FakeFallback, FakeJevClassifier

lanes = Lanes(auto=0.85, llm=0.60)
assert [lanes.lane_for(c) for c in (0.95, 0.70, 0.30)] == ["auto", "llm", "human"]

## 1. Define the state and providers

Declare `support_route` in your state: it stores the decision trace for the router named `support`. Each invocation starts a new trace; checkpoints retain prior snapshots. Fakes play the scripted responses in order, so rerun all cells to reset the tutorial.

In [ ]:
class State(TypedDict, total=False):
    messages: list[str]
    support_route: dict[str, Any]
    handled_by: str


classifier = FakeJevClassifier(
    Classification("orders", 0.95),
    Classification("orders", 0.70),
    Classification("orders", 0.30),
    Classification("orders", 0.70),
)
fallback = FakeFallback("billing", None)
router = JevRouter(
    name="support",
    routes={"orders": "Delivery and tracking", "billing": "Invoices and payments"},
    lanes=lanes,
    classifier=classifier,
    fallback=fallback,
)

## 2. Attach the router

Route names match handler node names by default. `targets=` can map them to other nodes, shared handlers or `END`. Human review needs a checkpointer and a stable `thread_id`. `InMemorySaver` is only for this process; use a persistent checkpointer for restart recovery.

In [ ]:
builder = StateGraph(State)
builder.add_node("intake", lambda state: {})
builder.add_edge(START, "intake")
for route in router.routes:
    builder.add_node(route, lambda state, route=route: {"handled_by": route})
    builder.add_edge(route, END)
router.attach(builder, after="intake")
graph = builder.compile(checkpointer=InMemorySaver())

## 3. Automatic and fallback decisions

The high-confidence case skips the fallback. The medium-confidence case accepts its valid route, even if it differs from the classifier. These assertions verify mechanics, not decision quality.

In [ ]:
auto = graph.invoke(
    {"messages": ["Where is my parcel?"]},
    {"configurable": {"thread_id": "auto"}},
)
assert auto["handled_by"] == "orders"
assert len(fallback.calls) == 0

llm = graph.invoke(
    {"messages": ["I was charged for an order I cancelled."]},
    {"configurable": {"thread_id": "llm"}},
)
assert llm["handled_by"] == "billing"
assert len(fallback.calls) == 1
print(llm["support_route"])

## 4. Pause, reject an invalid decision, and resume

In an application, show the pending review to a person before resuming. The case text remains in graph state; the interrupt contains route descriptions and suggestions. This tutorial scripts the human response. Resuming uses the **same config**, and must not classify again.

In [ ]:
config = {"configurable": {"thread_id": "human"}}
paused = graph.invoke({"messages": ["Please review my account and payment."]}, config)
assert "__interrupt__" in paused
assert "handled_by" not in paused
calls_before = len(classifier.calls)
print(paused["__interrupt__"][0].value)

try:
    graph.invoke(Command(resume="unknown"), config)
except ValidationError:
    print("Invalid route rejected; the review is still pending.")
else:
    raise AssertionError("Invalid route should be rejected")

resumed = graph.invoke(Command(resume="billing"), config)
assert resumed["handled_by"] == "billing"
assert len(classifier.calls) == calls_before
trace = RouteTrace.from_dict(resumed["support_route"])
assert [e.evaluator for e in trace.evaluations] == ["jev", "human"]
print(trace.to_dict())

## 5. A fallback can abstain

An abstention or unknown fallback route also requests human review. Provider exceptions instead propagate to the caller; they do not become low confidence.

In [ ]:
config = {"configurable": {"thread_id": "abstention"}}
paused = graph.invoke({"messages": ["Something is wrong with the order and payment."]}, config)
assert paused["__interrupt__"][0].value["reason"] == "llm_abstained"
resumed = graph.invoke(Command(resume="orders"), config)
trace = RouteTrace.from_dict(resumed["support_route"])
assert [e.evaluator for e in trace.evaluations] == ["jev", "llm", "human"]
assert len(classifier.calls) == 4
assert len(fallback.calls) == 2
print("Four cases completed; each classified once.")

## Next: bring your own keys

Replace the fakes with `JevClassifier()` and `LangChainFallback('openai:gpt-5-nano')` from `jev_lanes.adapters`. Install `langchain-openai` and set `TYPESAFE_API_KEY` and `OPENAI_API_KEY`; the provider example is `examples/byok.py`.

The router sends `messages` to TypeSafe, then to the fallback provider for medium-confidence cases. Supply `extract=` to restrict or redact that payload. Traces omit the case text, but graph checkpoints still contain the graph state.

Confidence is not accuracy. Choose thresholds using labeled calibration data and evaluate on a separate test split. The recorded synthetic smoke battery does not establish production accuracy or a benefit from fallback.